In [ ]:
# ============================================================================
# STAGE 4.1 -- SIMPLE WEIGHTING of the ensemble DAGs
#
# STANDALONE single cell. Reads the Stage 3 *_ensemble_dags.jsonl files and
# computes node scores W(v) in [0,1] using the SIMPLE weighting scheme from
# the paper (Sec. 3.4): every trace casts an equal vote, w_t = 1.
#
# Implements, per question:
#   - per-trace weight w_t = 1  (simple weighting)
#   - attestation masses  alpha^+(v), alpha^-(v)   (eq. 3)
#       alpha^+- (v) = (sum of w_t over support/defeat-attesting traces)
#                      / (sum of w_t over ALL traces M for that question)
#     Silent traces (never mention v) inflate the denominator -> lower confidence.
#   - weakest-link bundle strength  phi(S) = min_{u in S} W(u)   (eq. 4)
#   - gate-aware aggregate signals  Phi^+(v), Phi^-(v)           (eq. 5, 6)
#   - final node score
#       W(v) = max(0, alpha^+(v) * Phi^+(v) - alpha^-(v) * Phi^-(v))   (eq. 7)
#   - scores computed in TOPOLOGICAL order (predecessors before successors).
#
# No GPU / no model needed -- this is pure graph arithmetic over Stage 3 output.
#
# Input :  <DAG_DIR>/*_ensemble_dags.jsonl              (from Stage 3)
# Output:  <DAG_DIR>/*_ensemble_dags_weighted_simple.jsonl
#            same records, each ensemble node gains:
#              "W"           : final node score in [0,1]
#              "alpha_plus"  : positive attestation mass
#              "alpha_minus" : negative attestation mass
#              "phi_plus"    : Phi^+(v)
#              "phi_minus"   : Phi^-(v)
#            and each support/defeat bundle gains "phi" : phi(S).
# ============================================================================

import glob, json, os
from collections import defaultdict
from typing import Dict, List, Set, Tuple

# ----------------------------------------------------------------------------
# CONFIG -- adjust to match your setup.
# ----------------------------------------------------------------------------
DAG_DIR        = "./outputs_2234"
INPUT_SUFFIX   = "_ensemble_dags.jsonl"
OUTPUT_SUFFIX  = "_ensemble_dags_weighted_simple.jsonl"

INPUT_GLOB = os.path.join(DAG_DIR, "*" + INPUT_SUFFIX)


# ----------------------------------------------------------------------------
# Topological order over the ensemble DAG. Edges run predecessor -> node, where
# predecessors are the members of a node's support AND defeat bundles. Stage 3
# already guaranteed acyclicity (cycles were broken), but we Kahn-sort
# defensively and fall back to input order for any leftover nodes.
# ----------------------------------------------------------------------------
def topo_order(nodes: List[Dict]) -> List[str]:
    id_to_node = {n["id"]: n for n in nodes}

    def preds(n: Dict) -> Set[str]:
        out: Set[str] = set()
        for kind in ("support", "defeats"):
            for b in n.get(kind, []):
                for m in b.get("members", []):
                    if m in id_to_node:
                        out.add(m)
        return out

    indeg: Dict[str, int] = {nid: 0 for nid in id_to_node}
    children: Dict[str, List[str]] = defaultdict(list)
    for nid, n in id_to_node.items():
        for p in preds(n):
            children[p].append(nid)
            indeg[nid] += 1

    # Kahn -- seed with indegree-0 nodes in stable input order.
    queue = [n["id"] for n in nodes if indeg[n["id"]] == 0]
    order: List[str] = []
    qi = 0
    while qi < len(queue):
        u = queue[qi]; qi += 1
        order.append(u)
        for c in children[u]:
            indeg[c] -= 1
            if indeg[c] == 0:
                queue.append(c)

    # Any node not emitted (would only happen if a cycle survived) -- append
    # in input order so it still gets scored.
    if len(order) < len(nodes):
        seen = set(order)
        for n in nodes:
            if n["id"] not in seen:
                order.append(n["id"])
    return order


# ----------------------------------------------------------------------------
# Score one question's ensemble DAG in place. SIMPLE weighting: w_t = 1, so
# every mass is just (#attesting traces) / (#traces for this question).
# ----------------------------------------------------------------------------
def weight_simple(ensemble: Dict, n_source_traces: int) -> None:
    nodes = ensemble["nodes"]
    id_to_node = {n["id"]: n for n in nodes}

    # Denominator of eq. 3. With w_t = 1 this is |M| = number of traces that
    # went into this question's ensemble. Guard against a 0/empty group.
    M = max(1, int(n_source_traces))

    # W(v) values, filled in topological order.
    W: Dict[str, float] = {}

    for nid in topo_order(nodes):
        n = id_to_node[nid]
        gate = n.get("gate", "Atomic")

        # ---- attestation masses (eq. 3) --------------------------------
        # alpha^+ : fraction of traces (weight 1 each) that ASSERT v on the
        #           support side;  alpha^- : fraction on the defeat side.
        # For non-Atomic nodes this is the union of attesting traces across
        # the node's support / defeat bundles. For ATOMIC nodes there are no
        # support bundles (atomic premises), so alpha^+ must come from the
        # node's OWN attestation set (source_traces) -- i.e. how many traces
        # stated this fact. Without this, every Atomic Fact would get
        # alpha^+ = 0 and W = 0, zeroing out the whole DAG downstream.
        supp_traces: Set[str] = set()
        for b in n.get("support", []):
            supp_traces.update(b.get("attestations", []))
        def_traces: Set[str] = set()
        for b in n.get("defeats", []):
            def_traces.update(b.get("attestations", []))

        if gate == "Atomic":
            # node-level attestation: every trace that contributed this cluster
            supp_traces = set(n.get("source_traces", []))

        alpha_plus  = len(supp_traces) / M
        alpha_minus = len(def_traces)  / M

        # ---- weakest-link bundle strength phi(S) (eq. 4) ---------------
        # phi(S) = min over members u of W(u). Members are predecessors and,
        # in topological order, already scored. A missing member (dangling
        # ref) is treated as 0 -- it cannot lend strength.
        def phi(bundle: Dict) -> float:
            members = bundle.get("members", [])
            if not members:
                return 0.0
            return min(W.get(m, 0.0) for m in members)

        # annotate each bundle with its phi for downstream inspection / parsing
        for b in n.get("support", []):
            b["phi"] = phi(b)
        for b in n.get("defeats", []):
            b["phi"] = phi(b)

        # ---- gate-aware positive signal Phi^+(v) (eq. 5) ---------------
        sup = n.get("support", [])
        if gate == "Atomic":
            phi_plus = 1.0
        elif not sup:
            phi_plus = 0.0
        elif gate == "And":
            # exactly one support bundle
            phi_plus = sup[0]["phi"]
        else:  # Or
            phi_plus = max(b["phi"] for b in sup)

        # ---- gate-aware negative signal Phi^-(v) (eq. 6) ---------------
        defb = n.get("defeats", [])
        if not defb:
            phi_minus = 0.0
        elif len(defb) == 1:
            # paper: And-form defeater -> phi(S); otherwise max over defeaters.
            phi_minus = defb[0]["phi"]
        else:
            phi_minus = max(b["phi"] for b in defb)

        # ---- final node score (eq. 7) ----------------------------------
        score = max(0.0, alpha_plus * phi_plus - alpha_minus * phi_minus)
        W[nid] = score

        # write annotations back onto the node
        n["W"]           = score
        n["alpha_plus"]  = alpha_plus
        n["alpha_minus"] = alpha_minus
        n["phi_plus"]    = phi_plus
        n["phi_minus"]   = phi_minus

    ensemble["weighting_scheme"] = "simple"


# ----------------------------------------------------------------------------
# Driver -- stream each Stage 3 JSONL, weight every question, write a sibling
# *_weighted_simple.jsonl. Resume-aware: skips questions already in the output.
# ----------------------------------------------------------------------------
def already_done(out_path: str) -> Set[Tuple[str, str]]:
    done: Set[Tuple[str, str]] = set()
    if not os.path.exists(out_path):
        return done
    with open(out_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
                done.add((r["dataset"], r["question_id"]))
            except (json.JSONDecodeError, KeyError):
                continue
    return done


input_paths = sorted(p for p in glob.glob(INPUT_GLOB)
                     if not p.endswith(OUTPUT_SUFFIX))
if not input_paths:
    raise RuntimeError(f"No Stage 3 ensemble JSONLs at {INPUT_GLOB}. "
                       f"Did Stage 3 run?")

print(f"[stage 4.1] simple weighting over {len(input_paths)} ensemble JSONLs:")
for p in input_paths:
    print(f"  {p}")

global_q = 0
global_nodes = 0
for in_path in input_paths:
    out_path = in_path.replace(INPUT_SUFFIX, OUTPUT_SUFFIX)
    done = already_done(out_path)

    n_q = n_skip = n_nodes = 0
    # Append mode so resume works; if nothing is done it behaves like a fresh write.
    with open(in_path, "r", encoding="utf-8") as fin, \
         open(out_path, "a", encoding="utf-8") as fout:
        for line in fin:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            key = (rec.get("dataset", ""), rec.get("question_id", ""))
            if key in done:
                n_skip += 1
                continue

            ensemble = rec.get("ensemble_dag")
            if not isinstance(ensemble, dict) or "nodes" not in ensemble:
                rec.setdefault("weighting_errors", []).append("no ensemble_dag")
                fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                continue

            n_traces = rec.get("n_source_traces",
                               len(rec.get("source_traces", []) or []))
            weight_simple(ensemble, n_traces)

            # Convenience: net support per candidate answer cluster, recomputed
            # with the weighted node scores. With simple weighting this ranks
            # verdicts by W(Answer-node). Useful for the Stage 5 consensus pick.
            id_to_node = {n["id"]: n for n in ensemble["nodes"]}
            for ac in rec.get("answer_clusters", []) or []:
                node = id_to_node.get(ac.get("answer_cluster_id"))
                ac["W"] = node["W"] if node else 0.0
            if rec.get("answer_clusters"):
                rec["answer_clusters"].sort(
                    key=lambda x: (-x.get("W", 0.0), -x.get("n_attest", 0)))
                top = rec["answer_clusters"][0]
                rec["weighted_consensus_answer"]   = top.get("text")
                rec["weighted_consensus_score"]    = top.get("W", 0.0)

            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n_q += 1
            n_nodes += len(ensemble["nodes"])

    print(f"\n  {os.path.basename(in_path)} -> {os.path.basename(out_path)}")
    print(f"    questions weighted: {n_q}")
    print(f"    questions skipped (resume): {n_skip}")
    print(f"    ensemble nodes scored: {n_nodes}")
    global_q += n_q
    global_nodes += n_nodes

print(f"\n[stage 4.1] done. {global_q} questions, {global_nodes} nodes scored.")
print(f"[stage 4.1] outputs: {DAG_DIR}/*{OUTPUT_SUFFIX}")

# ----------------------------------------------------------------------------
# Quick sanity summary -- consensus-answer accuracy under simple weighting.
# (Compares weighted_consensus_answer to gold_label where both are present.)
# ----------------------------------------------------------------------------
print("\n[stage 4.1] consensus-answer check (weighted vs gold):")
for out_path in sorted(glob.glob(os.path.join(DAG_DIR, "*" + OUTPUT_SUFFIX))):
    n = correct = scored = 0
    with open(out_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            n += 1
            gold = (r.get("gold_label") or "").strip().lower()
            pred = (r.get("weighted_consensus_answer") or "").strip().lower()
            if gold and pred:
                scored += 1
                if gold == pred:
                    correct += 1
    acc = (100.0 * correct / scored) if scored else 0.0
    print(f"  {os.path.basename(out_path)}: {correct}/{scored} correct "
          f"({acc:.1f}%) over {n} questions")


In [ ]:
# ============================================================================
# STAGE 4.2 -- ACCURACY-WEIGHTED weighting of the ensemble DAGs
#
# STANDALONE single cell. Same scoring machinery as Stage 4.1, but the
# per-trace weight w_t is no longer uniform. Per the paper (Sec. 3.4):
#
#     "When per-model accuracies are available from a held-out test set,
#      we replace uniform votes with model-quality priors:
#          w_t = W_acc^{m(t)} ,  where  sum_m W_acc^m = 1.
#      Stronger models carry larger votes in both alpha^+(v) and alpha^-(v)."
#
# So this cell:
#   1. Computes each model's accuracy on the HELD-OUT split from the Stage 1
#      CSVs (predicted_label vs gold_label), per dataset.
#   2. Normalizes them per dataset so  sum_m W_acc^m = 1.
#   3. Re-runs the Sec. 3.4 weighting with w_t = W_acc^{model(t)}:
#        - alpha^+- (v) = (sum of w_t over attesting traces)
#                         / (sum of w_t over ALL traces M)         (eq. 3)
#        - phi(S) = min_{u in S} W(u)                               (eq. 4)
#        - gate-aware Phi^+(v), Phi^-(v)                            (eq. 5, 6)
#        - W(v) = max(0, alpha^+ * Phi^+ - alpha^- * Phi^-)         (eq. 7)
#      scored in topological order.
#
# No GPU / no model needed -- pure graph arithmetic + a CSV accuracy pass.
#
# Inputs:
#   <DAG_DIR>/*_ensemble_dags.jsonl          (from Stage 3)
#   <STAGE1_DIR>/*.csv                       (Stage 1 CSVs, for holdout accuracy)
# Output:
#   <DAG_DIR>/*_ensemble_dags_weighted_accuracy.jsonl
#     each ensemble node gains W / alpha_plus / alpha_minus / phi_plus /
#     phi_minus ; each bundle gains "phi" ; the record gains
#     "model_accuracy_weights" (the W_acc^m vector actually used).
# ============================================================================

import csv, glob, json, os
from collections import defaultdict
from typing import Dict, List, Set, Tuple

# ----------------------------------------------------------------------------
# CONFIG -- adjust to match your setup.
# ----------------------------------------------------------------------------
DAG_DIR        = "./outputs_2234"
STAGE1_DIR     = "./outputs_2234"

INPUT_SUFFIX   = "_ensemble_dags.jsonl"
OUTPUT_SUFFIX  = "_ensemble_dags_weighted_accuracy.jsonl"
INPUT_GLOB     = os.path.join(DAG_DIR, "*" + INPUT_SUFFIX)

# Where Stage 1 wrote per-trace CSVs. Stage 1 writes holdout traces to their
# OWN files -- "{dataset}_holdout_cots.csv" -- with the `split` column set to
# "holdout". Stage 2 explicitly skipped these, so holdout rows are NOT in the
# DAG pipeline and must be read straight from the Stage 1 holdout CSVs here.
# We glob the holdout files directly (filename is authoritative); the split
# column is double-checked below as a guard.
STAGE1_CSV_GLOB = os.path.join(STAGE1_DIR, "*_holdout_cots.csv")

# The `split` value Stage 1 stamps on holdout rows. Used only as a sanity
# guard -- the *_holdout_cots.csv filename is the real selector.
HOLDOUT_SPLIT_LABELS = {"holdout"}

# If a model has NO holdout rows for a dataset we can't measure its accuracy.
# FALLBACK_ACC is used so it still gets a (small, uniform) vote rather than 0.
FALLBACK_ACC = 0.5

# Optional manual override. If you already know the per-model holdout
# accuracies, put them here as {dataset: {model: raw_accuracy}} and the CSV
# pass is skipped for those datasets. Leave empty to compute from CSVs.
MANUAL_ACCURACIES: Dict[str, Dict[str, float]] = {
    # "sara": {"qwen2.5-7b-instruct": 0.81, "gemma-3-12b-instruct": 0.79, ...},
}


# ----------------------------------------------------------------------------
# 1. Per-model holdout accuracy from the Stage 1 CSVs.
#    accuracy(model, dataset) = correct holdout rows / total holdout rows.
#    A "correct" row has predicted_label == gold_label (normalized).
# ----------------------------------------------------------------------------
def _norm(s: str) -> str:
    return (s or "").strip().lower()


def compute_holdout_accuracies() -> Dict[str, Dict[str, float]]:
    """Returns {dataset: {model: raw_accuracy}} from holdout-split CSV rows."""
    # (dataset, model) -> [n_correct, n_total]
    tally: Dict[Tuple[str, str], List[int]] = defaultdict(lambda: [0, 0])

    csv_paths = sorted(glob.glob(STAGE1_CSV_GLOB))
    if not csv_paths:
        print(f"[stage 4.2] WARNING: no Stage 1 holdout CSVs matching "
              f"{STAGE1_CSV_GLOB}. Expected files like sara_holdout_cots.csv. "
              f"Without them, accuracy weighting falls back to uniform "
              f"(equivalent to Stage 4.1). Set MANUAL_ACCURACIES or fix the "
              f"path/filenames.")
        return {}

    print(f"[stage 4.2] scanning {len(csv_paths)} Stage 1 CSVs for holdout rows:")
    for p in csv_paths:
        n_holdout = 0
        with open(p, "r", encoding="utf-8", newline="") as f:
            reader = csv.DictReader(f)
            for row in reader:
                # Filename already guarantees these are holdout rows. If the
                # split column is present we still check it; if it's blank we
                # accept the row (filename is authoritative).
                split = _norm(row.get("split"))
                if split and split not in HOLDOUT_SPLIT_LABELS:
                    continue
                ds  = _norm(row.get("dataset"))
                mdl = (row.get("model") or "").strip()
                pred = _norm(row.get("predicted_label"))
                gold = _norm(row.get("gold_label"))
                if not (ds and mdl and pred and gold):
                    continue
                n_holdout += 1
                t = tally[(ds, mdl)]
                t[1] += 1
                if pred == gold:
                    t[0] += 1
        print(f"  {os.path.basename(p)}: {n_holdout} holdout rows")

    accs: Dict[str, Dict[str, float]] = defaultdict(dict)
    for (ds, mdl), (correct, total) in tally.items():
        accs[ds][mdl] = (correct / total) if total else FALLBACK_ACC
    return accs


def normalized_weights(raw: Dict[str, float],
                        all_models: Set[str]) -> Dict[str, float]:
    """Normalize raw accuracies so they sum to 1 over all_models (paper:
    sum_m W_acc^m = 1). Models with no measured accuracy get FALLBACK_ACC
    before normalization."""
    filled = {m: raw.get(m, FALLBACK_ACC) for m in all_models}
    total = sum(filled.values())
    if total <= 0:
        # degenerate -- fall back to uniform
        u = 1.0 / max(1, len(all_models))
        return {m: u for m in all_models}
    return {m: v / total for m, v in filled.items()}


# ----------------------------------------------------------------------------
# trace_key -> model name. Stage 3 built trace keys as "{model}::{sample_idx}".
# ----------------------------------------------------------------------------
def model_of_trace(trace_key: str) -> str:
    return trace_key.rsplit("::", 1)[0]


# ----------------------------------------------------------------------------
# 2. Topological order over the ensemble DAG (predecessors before node).
# ----------------------------------------------------------------------------
def topo_order(nodes: List[Dict]) -> List[str]:
    id_to_node = {n["id"]: n for n in nodes}

    def preds(n: Dict) -> Set[str]:
        out: Set[str] = set()
        for kind in ("support", "defeats"):
            for b in n.get(kind, []):
                for m in b.get("members", []):
                    if m in id_to_node:
                        out.add(m)
        return out

    indeg = {nid: 0 for nid in id_to_node}
    children: Dict[str, List[str]] = defaultdict(list)
    for nid, n in id_to_node.items():
        for p in preds(n):
            children[p].append(nid)
            indeg[nid] += 1

    queue = [n["id"] for n in nodes if indeg[n["id"]] == 0]
    order, qi = [], 0
    while qi < len(queue):
        u = queue[qi]; qi += 1
        order.append(u)
        for c in children[u]:
            indeg[c] -= 1
            if indeg[c] == 0:
                queue.append(c)
    if len(order) < len(nodes):  # leftover (cycle survived) -- append stably
        seen = set(order)
        order += [n["id"] for n in nodes if n["id"] not in seen]
    return order


# ----------------------------------------------------------------------------
# 3. Accuracy-weighted scoring of one question's ensemble DAG, in place.
#    w_t = W_acc^{model(t)}.  Masses are weight-sums, not plain counts.
# ----------------------------------------------------------------------------
def weight_accuracy(ensemble: Dict,
                    source_traces: List[str],
                    w_acc: Dict[str, float]) -> None:
    nodes = ensemble["nodes"]
    id_to_node = {n["id"]: n for n in nodes}

    # Per-trace weight w_t for every trace in this question's M set.
    # A trace whose model is missing from w_acc falls back to FALLBACK_ACC.
    w_t: Dict[str, float] = {}
    for tk in source_traces:
        w_t[tk] = w_acc.get(model_of_trace(tk), FALLBACK_ACC)

    # Denominator of eq. 3: sum of w_t over ALL traces M for this question.
    denom = sum(w_t.values())
    if denom <= 0:
        denom = 1.0

    W: Dict[str, float] = {}

    for nid in topo_order(nodes):
        n = id_to_node[nid]
        gate = n.get("gate", "Atomic")

        # ---- attestation masses (eq. 3) --------------------------------
        # Numerator = sum of w_t over the traces attesting the node on the
        # support / defeat side. Stronger models contribute larger w_t.
        supp_traces: Set[str] = set()
        for b in n.get("support", []):
            supp_traces.update(b.get("attestations", []))
        def_traces: Set[str] = set()
        for b in n.get("defeats", []):
            def_traces.update(b.get("attestations", []))

        if gate == "Atomic":
            # Atomic premises have no support bundles -- alpha^+ comes from the
            # node's own attestation set (same fix as Stage 4.1). Otherwise
            # every Atomic Fact would get alpha^+ = 0 and zero the DAG.
            supp_traces = set(n.get("source_traces", []))

        alpha_plus  = sum(w_t.get(t, FALLBACK_ACC) for t in supp_traces) / denom
        alpha_minus = sum(w_t.get(t, FALLBACK_ACC) for t in def_traces)  / denom

        # ---- weakest-link bundle strength phi(S) (eq. 4) ---------------
        def phi(bundle: Dict) -> float:
            members = bundle.get("members", [])
            if not members:
                return 0.0
            return min(W.get(m, 0.0) for m in members)

        for b in n.get("support", []):
            b["phi"] = phi(b)
        for b in n.get("defeats", []):
            b["phi"] = phi(b)

        # ---- gate-aware positive signal Phi^+(v) (eq. 5) ---------------
        sup = n.get("support", [])
        if gate == "Atomic":
            phi_plus = 1.0
        elif not sup:
            phi_plus = 0.0
        elif gate == "And":
            phi_plus = sup[0]["phi"]
        else:  # Or
            phi_plus = max(b["phi"] for b in sup)

        # ---- gate-aware negative signal Phi^-(v) (eq. 6) ---------------
        defb = n.get("defeats", [])
        if not defb:
            phi_minus = 0.0
        elif len(defb) == 1:
            phi_minus = defb[0]["phi"]
        else:
            phi_minus = max(b["phi"] for b in defb)

        # ---- final node score (eq. 7) ----------------------------------
        score = max(0.0, alpha_plus * phi_plus - alpha_minus * phi_minus)
        W[nid] = score

        n["W"]           = score
        n["alpha_plus"]  = alpha_plus
        n["alpha_minus"] = alpha_minus
        n["phi_plus"]    = phi_plus
        n["phi_minus"]   = phi_minus

    ensemble["weighting_scheme"] = "accuracy"


# ----------------------------------------------------------------------------
# Driver.
# ----------------------------------------------------------------------------
def already_done(out_path: str) -> Set[Tuple[str, str]]:
    done: Set[Tuple[str, str]] = set()
    if not os.path.exists(out_path):
        return done
    with open(out_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
                done.add((r["dataset"], r["question_id"]))
            except (json.JSONDecodeError, KeyError):
                continue
    return done


# --- compute (or load) per-dataset model accuracies -------------------------
computed = compute_holdout_accuracies()
# manual overrides win
raw_accs: Dict[str, Dict[str, float]] = defaultdict(dict)
for ds, d in computed.items():
    raw_accs[ds].update(d)
for ds, d in MANUAL_ACCURACIES.items():
    raw_accs[_norm(ds)].update(d)

print("\n[stage 4.2] raw per-model holdout accuracies:")
if not raw_accs:
    print("  (none found -- all datasets will fall back to uniform weighting)")
for ds in sorted(raw_accs):
    for m in sorted(raw_accs[ds]):
        print(f"  [{ds}] {m}: {raw_accs[ds][m]:.4f}")

# --- process each ensemble JSONL --------------------------------------------
input_paths = sorted(p for p in glob.glob(INPUT_GLOB)
                     if not p.endswith(OUTPUT_SUFFIX))
if not input_paths:
    raise RuntimeError(f"No Stage 3 ensemble JSONLs at {INPUT_GLOB}. "
                       f"Did Stage 3 run?")

print(f"\n[stage 4.2] accuracy weighting over {len(input_paths)} ensemble JSONLs:")
for p in input_paths:
    print(f"  {p}")

global_q = global_nodes = 0
for in_path in input_paths:
    out_path = in_path.replace(INPUT_SUFFIX, OUTPUT_SUFFIX)
    done = already_done(out_path)

    # First pass over this file: collect the set of models that actually
    # appear, so the W_acc vector is normalized over exactly those models.
    ds_models: Dict[str, Set[str]] = defaultdict(set)
    with open(in_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue
            ds = _norm(rec.get("dataset"))
            for tk in rec.get("source_traces", []) or []:
                ds_models[ds].add(model_of_trace(tk))

    # Build the normalized W_acc vector per dataset present in this file.
    norm_w: Dict[str, Dict[str, float]] = {}
    for ds, models in ds_models.items():
        norm_w[ds] = normalized_weights(raw_accs.get(ds, {}), models)
        pretty = ", ".join(f"{m}={norm_w[ds][m]:.3f}" for m in sorted(models))
        print(f"\n  [{ds}] normalized W_acc (sum=1): {pretty}")

    n_q = n_skip = n_nodes = 0
    with open(in_path, "r", encoding="utf-8") as fin, \
         open(out_path, "a", encoding="utf-8") as fout:
        for line in fin:
            try:
                rec = json.loads(line)
            except json.JSONDecodeError:
                continue

            key = (rec.get("dataset", ""), rec.get("question_id", ""))
            if key in done:
                n_skip += 1
                continue

            ensemble = rec.get("ensemble_dag")
            if not isinstance(ensemble, dict) or "nodes" not in ensemble:
                rec.setdefault("weighting_errors", []).append("no ensemble_dag")
                fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
                continue

            ds = _norm(rec.get("dataset"))
            source_traces = rec.get("source_traces", []) or []
            w_acc = norm_w.get(ds, {})

            weight_accuracy(ensemble, source_traces, w_acc)

            # record the W_acc vector actually applied (only the models used)
            used_models = {model_of_trace(tk) for tk in source_traces}
            rec["model_accuracy_weights"] = {m: w_acc.get(m, FALLBACK_ACC)
                                             for m in sorted(used_models)}

            # weighted net support per candidate answer cluster
            id_to_node = {n["id"]: n for n in ensemble["nodes"]}
            for ac in rec.get("answer_clusters", []) or []:
                node = id_to_node.get(ac.get("answer_cluster_id"))
                ac["W"] = node["W"] if node else 0.0
            if rec.get("answer_clusters"):
                rec["answer_clusters"].sort(
                    key=lambda x: (-x.get("W", 0.0), -x.get("n_attest", 0)))
                top = rec["answer_clusters"][0]
                rec["weighted_consensus_answer"] = top.get("text")
                rec["weighted_consensus_score"]  = top.get("W", 0.0)

            fout.write(json.dumps(rec, ensure_ascii=False) + "\n")
            n_q += 1
            n_nodes += len(ensemble["nodes"])

    print(f"\n  {os.path.basename(in_path)} -> {os.path.basename(out_path)}")
    print(f"    questions weighted: {n_q}")
    print(f"    questions skipped (resume): {n_skip}")
    print(f"    ensemble nodes scored: {n_nodes}")
    global_q += n_q
    global_nodes += n_nodes

print(f"\n[stage 4.2] done. {global_q} questions, {global_nodes} nodes scored.")
print(f"[stage 4.2] outputs: {DAG_DIR}/*{OUTPUT_SUFFIX}")

# ----------------------------------------------------------------------------
# Consensus-answer accuracy under accuracy weighting.
# ----------------------------------------------------------------------------
print("\n[stage 4.2] consensus-answer check (weighted vs gold):")
for out_path in sorted(glob.glob(os.path.join(DAG_DIR, "*" + OUTPUT_SUFFIX))):
    n = correct = scored = 0
    with open(out_path, "r", encoding="utf-8") as f:
        for line in f:
            try:
                r = json.loads(line)
            except json.JSONDecodeError:
                continue
            n += 1
            gold = (r.get("gold_label") or "").strip().lower()
            pred = (r.get("weighted_consensus_answer") or "").strip().lower()
            if gold and pred:
                scored += 1
                if gold == pred:
                    correct += 1
    acc = (100.0 * correct / scored) if scored else 0.0
    print(f"  {os.path.basename(out_path)}: {correct}/{scored} correct "
          f"({acc:.1f}%) over {n} questions")
